# Aula 7 · Onde está e quantos são: detecção de objetos com YOLO

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tassid/pdi-agro/blob/main/notebooks/Aula7_YOLO_Deteccao.ipynb)

Material do módulo YOLO do **Pixel no Campo** (PPGTCA/UTFPR Medianeira). Companheiro dos labs 7.1 a 7.5 do site.

**Hoje, no Colab:**

1. Rodar um YOLO pronto (`yolo26n.pt`, treinado no COCO), ler a saída e contar objetos.
2. Escrever IoU e NMS à mão e conferir com a `torchvision`.
3. Gerar um piquete visto de drone, salvar os rótulos no formato YOLO, desenhar de volta e ver o modelo do COCO falhar.

In [ ]:
# Ambiente de execução → Alterar o tipo de ambiente → GPU (T4). Depois rode esta célula.
!pip install -q -U ultralytics
import ultralytics
ultralytics.checks()

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt

def mostrar(*imgs, titulos=None, tam=5):
    """mostra imagens BGR (do OpenCV) lado a lado"""
    fig, ax = plt.subplots(1, len(imgs), figsize=(tam * len(imgs), tam))
    ax = np.atleast_1d(ax)
    for i, im in enumerate(imgs):
        ax[i].imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax[i].axis("off")
        if titulos: ax[i].set_title(titulos[i])
    plt.show()

## 1. Um YOLO pronto

O `yolo26n.pt` foi treinado no banco COCO, com 80 classes do dia a dia (pessoa, carro, cachorro, *cow*, *sheep*, *horse*, *bird*...). Na primeira vez, o ultralytics baixa os pesos sozinho.

In [ ]:
from ultralytics import YOLO
from ultralytics.utils import ASSETS              # fotos de exemplo que vêm junto com o pacote

modelo = YOLO("yolo26n.pt")                     # n = nano. Troque por yolo26s.pt para ver a diferença de velocidade e acerto
res = modelo.predict(ASSETS / "bus.jpg", conf=0.25, imgsz=640)[0]
mostrar(res.plot(), titulos=["saída do yolo26n"])

### Lendo a saída

Cada `res` tem `res.boxes`, com uma linha por objeto. Os mesmos números em formatos diferentes (Lab 7.2):

| atributo | formato |
|---|---|
| `xyxy` | x0, y0, x1, y1 em pixels da foto original |
| `xywh` | centro x, centro y, largura, altura em pixels |
| `xywhn` | o mesmo, dividido pela largura e altura da foto: é a linha do rótulo YOLO |
| `conf` | confiança, de 0 a 1 |
| `cls` | índice da classe; o nome está em `res.names` |

In [ ]:
import pandas as pd
linhas = []
for b in res.boxes:
    x0, y0, x1, y1 = b.xyxy[0].tolist()
    cx, cy, w, h = b.xywhn[0].tolist()
    linhas.append(dict(classe=res.names[int(b.cls)], conf=round(float(b.conf), 3),
                       x0=round(x0), y0=round(y0), x1=round(x1), y1=round(y1),
                       rotulo_yolo=f"{int(b.cls)} {cx:.4f} {cy:.4f} {w:.4f} {h:.4f}"))
tabela = pd.DataFrame(linhas)
print(tabela.to_string(index=False))
print("\ncontagem por classe:\n", tabela.classe.value_counts().to_string())

### Agora com uma foto sua

Uma foto de vacas, cavalos, galinhas, um pátio com carros. Rode a célula, escolha o arquivo e veja a contagem. Mude `conf` para 0,1 e 0,6 e compare.

In [ ]:
try:
    from google.colab import files
    enviados = files.upload()
    caminho = next(iter(enviados))
except Exception:                                # fora do Colab: use qualquer caminho local
    caminho = str(ASSETS / "zidane.jpg")

for conf in [0.1, 0.25, 0.6]:
    r = modelo.predict(caminho, conf=conf, verbose=False)[0]
    nomes = [r.names[int(c)] for c in r.boxes.cls]
    print(f"conf={conf}: {len(nomes)} objetos", {n: nomes.count(n) for n in set(nomes)})
mostrar(modelo.predict(caminho, conf=0.25, verbose=False)[0].plot())

## 2. IoU e NMS à mão

Antes de virarem a resposta final, as caixas de um detector clássico passam pelo NMS (Lab 7.5). Vamos escrever as duas funções e conferir com a implementação da `torchvision`.

Para ter candidatas parecidas com a saída bruta de um detector, pegamos as caixas finais e geramos várias versões levemente deslocadas de cada uma, com confianças menores.

In [ ]:
def iou(a, b):                                   # caixas x0, y0, x1, y1
    ix0, iy0 = max(a[0], b[0]), max(a[1], b[1])
    ix1, iy1 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix1 - ix0) * max(0, iy1 - iy0)
    area = lambda c: (c[2] - c[0]) * (c[3] - c[1])
    return inter / (area(a) + area(b) - inter)

# teste do quiz: duas caixas 40x20, uma deslocada 10 px
print("IoU =", iou([0, 0, 40, 20], [10, 0, 50, 20]))

def nms(caixas, notas, limiar=0.7):
    ordem = list(np.argsort(notas)[::-1])        # da maior confiança para a menor
    manter = []
    while ordem:
        i = ordem.pop(0)
        manter.append(i)
        ordem = [j for j in ordem if iou(caixas[i], caixas[j]) <= limiar]
    return manter

rng = np.random.default_rng(0)
finais = res.boxes.xyxy.cpu().numpy()
caixas, notas = [], []
for b, c in zip(finais, res.boxes.conf.cpu().numpy()):
    w, h = b[2] - b[0], b[3] - b[1]
    for k in range(5):                           # 5 candidatas por objeto
        d = rng.normal(0, 0.04 * (k > 0), 4) * [w, h, w, h]
        caixas.append((b + d).tolist()); notas.append(float(c) * (1 - 0.15 * k))
print(len(caixas), "candidatas para", len(finais), "objetos")

In [ ]:
import torch, torchvision
for limiar in [0.3, 0.5, 0.7, 0.9]:
    nosso = sorted(nms(caixas, notas, limiar))
    tv = sorted(torchvision.ops.nms(torch.tensor(caixas), torch.tensor(notas), limiar).tolist())
    print(f"limiar {limiar}: {len(nosso):3d} caixas mantidas · igual à torchvision? {nosso == tv}")

**Pra pensar:** com limiar 0,9 sobram duplicatas; com 0,3 objetos vizinhos de verdade (duas pessoas lado a lado) podem sumir. É o mesmo efeito do rebanho aglomerado no Lab 7.5.

> O YOLO26 é treinado para que só um ponto da grade responda por objeto, então a saída dele já vem sem duplicatas. O NMS continua sendo usado nas versões anteriores (v8, 11) e em quase todo detector que vocês vão encontrar em artigos.

## 3. O boi visto de cima

A função abaixo gera um piquete sintético visto de drone, parecido com o dos labs do site, e devolve a imagem e as caixas verdadeiras. Na Aula 8 ela vai gerar o banco inteiro de treino.

In [ ]:
import numpy as np, cv2

def _ruido(rng, h, w, escala):
    """ruído suave: sorteia uma grade pequena e amplia com interpolação cúbica"""
    g = rng.random((max(2, h // escala), max(2, w // escala))).astype(np.float32)
    return cv2.resize(g, (w, h), interpolation=cv2.INTER_CUBIC).clip(0, 1)

def gerar_piquete(semente, fazenda="A", n=None, tam=640, luz=1.0, sol=(5, 4)):
    """Gera um piquete visto de drone e as caixas verdadeiras dos animais.
    fazenda A: nelore claro em pasto verde.  fazenda B: gado escuro, solo exposto, mais cupinzeiros.
    Devolve (imagem BGR uint8, lista de caixas [x0, y0, x1, y1] em pixels)."""
    rng = np.random.default_rng(semente)
    H = W = tam
    B = fazenda == "B"
    n = n if n is not None else int(rng.integers(8, 22))
    # capim: verde com manchas secas
    n1, n2 = _ruido(rng, H, W, 80), _ruido(rng, H, W, 12)
    seco = np.clip((_ruido(rng, H, W, 140) - (0.35 if B else 0.55)) * 3, 0, 1)
    img = np.stack([52 + 18 * n1 + 8 * n2 + 10 * seco,            # B
                    118 + 36 * n1 + 20 * n2 + 30 * seco,           # G
                    78 + 40 * n1 + 18 * n2 + 70 * seco], -1)       # R
    if B:  # solo vermelho aparecendo
        solo = np.clip((_ruido(rng, H, W, 60) - 0.6) * 5, 0, 1)[..., None]
        img = img * (1 - solo) + np.array([48, 74, 150]) * solo
    img = img.astype(np.float32)
    sx, sy = sol
    # árvores (sombra agora, copa por último)
    arvores = [(int(rng.integers(60, W - 60)), int(rng.integers(60, H - 60)), int(rng.integers(40, 60))) for _ in range(int(rng.integers(1, 3)))]
    sombra = np.zeros((H, W), np.uint8)
    for x, y, r in arvores:
        cv2.circle(sombra, (x + 4 * sx, y + 4 * sy), r, 1, -1)
    # cupinzeiros
    cupins = []
    for _ in range(int(rng.integers(9, 13)) if B else int(rng.integers(3, 5))):
        cupins.append((int(rng.integers(20, W - 20)), int(rng.integers(20, H - 20)), int(rng.integers(7, 11))))
    for x, y, r in cupins:
        cv2.circle(sombra, (x + sx // 2, y + sy // 2), r, 1, -1)
    # animais
    animais, tent = [], 0
    while len(animais) < n and tent < 2000:
        tent += 1
        x, y = rng.uniform(28, W - 28), rng.uniform(28, H - 28)
        bez = rng.random() < 0.18
        a = (10.5 if bez else 16) + rng.random() * 2.4
        b = (5.2 if bez else 7.2) + rng.random() * 1.2
        if any(np.hypot(o[0] - x, o[1] - y) < 34 for o in animais):
            continue
        if any(np.hypot(c[0] - x, c[1] - y) < 26 for c in cupins):
            continue
        ang = rng.random() * 360
        escuro = B and rng.random() < 0.6     # na fazenda A só tem nelore claro
        cor = (np.array([30, 40, 58]) if rng.random() < 0.5 else np.array([36, 38, 44])) if escuro else np.array([206, 218, 224]) + rng.random(3) * 16
        animais.append((x, y, a, b, ang, cor))
    for x, y, a, b, ang, cor in animais:
        cv2.ellipse(sombra, (int(x + sx), int(y + sy)), (int(a), int(b)), ang, 0, 360, 1, -1)
    img[sombra > 0] *= 0.56
    for x, y, r in cupins:
        cv2.circle(img, (x, y), r, (72, 98, 138) if B else (132, 160, 176), -1)
    caixas = []
    for x, y, a, b, ang, cor in animais:
        cv2.ellipse(img, (int(x), int(y)), (int(a), int(b)), ang, 0, 360, cor.tolist(), -1)
        t = np.deg2rad(ang)
        hx, hy = x + 1.1 * a * np.cos(t), y + 1.1 * a * np.sin(t)
        cv2.ellipse(img, (int(hx), int(hy)), (int(0.3 * a) + 1, int(0.55 * b) + 1), ang, 0, 360, (cor * 0.92).tolist(), -1)
        # caixa: elipse do corpo com a cabeça até 1,4·a à frente
        ex = np.sqrt((1.4 * a * np.cos(t)) ** 2 + (b * np.sin(t)) ** 2)
        ey = np.sqrt((1.4 * a * np.sin(t)) ** 2 + (b * np.cos(t)) ** 2)
        caixas.append([max(0, x - ex), max(0, y - ey), min(W, x + ex), min(H, y + ey)])
    # copa das árvores por cima (animal debaixo fica parcialmente escondido, mas continua rotulado)
    copa = np.zeros((H, W), np.uint8)
    for x, y, r in arvores:
        cv2.circle(copa, (x, y), int(r * 0.7), 1, -1)
    folhas = _ruido(rng, H, W, 6)
    m = copa > 0
    img[m] = np.stack([30 + 14 * folhas, 72 + 34 * folhas, 34 + 30 * folhas], -1)[m]
    # quanto de cada animal ficou visível? some do rótulo se a copa cobrir mais de 70%
    vis = []
    for (x0, y0, x1, y1) in caixas:
        sub = copa[int(y0):int(y1) + 1, int(x0):int(x1) + 1]
        vis.append(1 - sub.mean() if sub.size else 1)
    caixas = [c for c, v in zip(caixas, vis) if v > 0.3]
    img = img * luz + rng.normal(0, 3, img.shape)
    return np.clip(img, 0, 255).astype(np.uint8), caixas

def linhas_yolo(caixas, W, H, classe=0):
    out = []
    for x0, y0, x1, y1 in caixas:
        out.append(f"{classe} {(x0 + x1) / 2 / W:.6f} {(y0 + y1) / 2 / H:.6f} {(x1 - x0) / W:.6f} {(y1 - y0) / H:.6f}")
    return out

def desenhar_rotulos(img, txt, cor=(0, 224, 255)):
    """lê um .txt no formato YOLO e desenha as caixas de volta na imagem (o teste que todo banco deve passar)"""
    H, W = img.shape[:2]
    out = img.copy()
    for linha in open(txt).read().split("\n"):
        if not linha.strip():
            continue
        c, cx, cy, w, h = map(float, linha.split())
        x0, y0, x1, y1 = (cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H
        cv2.rectangle(out, (int(x0), int(y0)), (int(x1), int(y1)), cor, 2)
    return out

In [ ]:
import os
os.makedirs("piquete", exist_ok=True)
img, caixas = gerar_piquete(7, "A")
cv2.imwrite("piquete/voo01_0001.jpg", img)
open("piquete/voo01_0001.txt", "w").write("\n".join(linhas_yolo(caixas, img.shape[1], img.shape[0])) + "\n")
print(open("piquete/voo01_0001.txt").read())
mostrar(img, desenhar_rotulos(img, "piquete/voo01_0001.txt"), titulos=["piquete", "rótulos desenhados de volta"])

Sempre desenhe os rótulos de volta antes de treinar. Se as caixas aparecerem deslocadas ou do tamanho errado, o formato está trocado (Lab 8.1).

### E o modelo do COCO?

In [ ]:
r = modelo.predict(img, conf=0.1, verbose=False)[0]
nomes = [r.names[int(c)] for c in r.boxes.cls]
print("animais de verdade:", len(caixas))
print("o que o yolo26n do COCO achou (conf ≥ 0,1):", {n: nomes.count(n) for n in set(nomes)} or "nada")
mostrar(r.plot(), titulos=["yolo26n (COCO) no piquete visto de cima"])

Nos nossos testes, o `yolo26n` do COCO não achou nenhuma vaca: chamou os bois de *kite* (pipa) e *bird* (pássaro). Faz sentido: o COCO tem a classe *cow*, mas quase só com fotos de lado, na altura dos olhos. Visto de cima, o boi vira uma elipse clara contra o verde, e o que o modelo conhece com essa cara, visto de baixo para cima, é pipa no céu. É o atalho da Aula 6: ele aprendeu "vaca de perfil com pernas", não "vaca". **Na Aula 8 vamos treinar com o nosso ponto de vista.**

## Desafios

1. **Aquecimento.** Gere 3 piquetes com `gerar_piquete(s, "B")` e conte quantos animais escuros e quantos cupinzeiros aparecem. Por que eles são difíceis para um modelo treinado só na fazenda A?
2. **Médio.** Escreva `coco_para_yolo(json, pasta_saida)` que converte um JSON no formato COCO (`bbox = [x, y, largura, altura]`) para os .txt do YOLO. Teste com `desenhar_rotulos`.
3. **Desafio.** Rode o `yolo26n.pt` num vídeo seu com `conf` em 0,1, 0,25 e 0,5. Monte uma tabela com a contagem média por quadro e explique o padrão.